# 04 - Silver -> Gold (modelo dimensional, Ralph Kimball)

Esquema estrela para as avaliações do Metacritic:

```
DIM_GAME  DIM_PLATFORM  DIM_GENRE  DIM_DEVELOPER  DIM_PUBLISHER  DIM_DATE
      \         \          |           |             /           /
                      FACT_GAME_REVIEW
          (grão: 1 linha por jogo/plataforma | medidas: METASCORE, USER_SCORE)
```

Cada dimensão tem uma *surrogate key* inteira (`<DIM>_SK`) e o membro `-1 = NÃO INFORMADO`, usado quando a chave estrangeira da fato é nula.

In [ ]:
from pyspark.sql import DataFrame, Window, functions as F

CATALOG       = "workspace"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA   = "gold"

UNKNOWN_SK   = -1                 # membro "desconhecido" de cada dimensão (FK nula na fato)
UNKNOWN_NAME = "NÃO INFORMADO"


def build_dimension(df: DataFrame, natural_col: str, dim_name: str) -> DataFrame:
    """Dimensão simples: <DIM>_SK sequencial + <DIM>_NAME (chave natural) + membro -1."""
    sk, nm = f"{dim_name}_SK", f"{dim_name}_NAME"
    dim = (df.where(F.col(natural_col).isNotNull())
             .select(F.col(natural_col).alias(nm)).distinct()
             .withColumn(sk, F.row_number().over(Window.orderBy(nm)).cast("int"))
             .select(sk, nm))
    unknown = spark.range(1).select(F.lit(UNKNOWN_SK).alias(sk), F.lit(UNKNOWN_NAME).alias(nm))
    return dim.unionByName(unknown).withColumn("FILEDATE_GOLD", F.current_timestamp())


def build_dim_game(df: DataFrame) -> DataFrame:
    """DIM_GAME: um registro por ID do Metacritic (jogo + plataforma)."""
    dim = (df.select("ID", "NAME").distinct()
             .withColumn("GAME_SK", F.row_number().over(Window.orderBy("ID")).cast("int"))
             .select("GAME_SK", F.col("ID").alias("GAME_ID"), F.col("NAME").alias("GAME_NAME")))
    unknown = spark.range(1).select(
        F.lit(UNKNOWN_SK).alias("GAME_SK"),
        F.lit(None).cast("int").alias("GAME_ID"),
        F.lit(UNKNOWN_NAME).alias("GAME_NAME"))
    return dim.unionByName(unknown).withColumn("FILEDATE_GOLD", F.current_timestamp())


def build_dim_date(df: DataFrame, date_col: str = "RELEASE_DATE") -> DataFrame:
    """DIM_DATE: calendário completo entre a menor e a maior data de lançamento."""
    bounds = df.agg(F.min(date_col).alias("d0"), F.max(date_col).alias("d1")).first()
    dim = (spark.range(1)
           .select(F.explode(F.sequence(F.lit(bounds["d0"]), F.lit(bounds["d1"]))).alias("DATE"))
           .select(
               F.date_format("DATE", "yyyyMMdd").cast("int").alias("DATE_SK"),
               F.col("DATE"),
               F.year("DATE").alias("YEAR"),
               F.quarter("DATE").alias("QUARTER"),
               F.month("DATE").alias("MONTH"),
               F.date_format("DATE", "MMMM").alias("MONTH_NAME"),
               F.dayofmonth("DATE").alias("DAY"),
               F.dayofweek("DATE").alias("DAY_OF_WEEK"),
               F.date_format("DATE", "EEEE").alias("DAY_NAME"),
               F.dayofweek("DATE").isin(1, 7).alias("IS_WEEKEND"),
           ))
    unknown = spark.range(1).select(
        F.lit(UNKNOWN_SK).alias("DATE_SK"),
        F.lit(None).cast("date").alias("DATE"),
        F.lit(None).cast("int").alias("YEAR"),
        F.lit(None).cast("int").alias("QUARTER"),
        F.lit(None).cast("int").alias("MONTH"),
        F.lit(UNKNOWN_NAME).alias("MONTH_NAME"),
        F.lit(None).cast("int").alias("DAY"),
        F.lit(None).cast("int").alias("DAY_OF_WEEK"),
        F.lit(UNKNOWN_NAME).alias("DAY_NAME"),
        F.lit(None).cast("boolean").alias("IS_WEEKEND"))
    return dim.unionByName(unknown).withColumn("FILEDATE_GOLD", F.current_timestamp())


def attach_sk(df: DataFrame, dim: DataFrame, natural_col: str, dim_name: str) -> DataFrame:
    """Troca a chave natural pela surrogate key da dimensão (sem correspondência -> -1)."""
    sk, nm = f"{dim_name}_SK", f"{dim_name}_NAME"
    lookup = dim.select(F.col(sk), F.col(nm).alias("_natural_key"))
    return (df.join(lookup, df[natural_col] == lookup["_natural_key"], "left")
              .withColumn(sk, F.coalesce(F.col(sk), F.lit(UNKNOWN_SK)))
              .drop("_natural_key"))


def build_fact_game_review(df, dim_game, dim_platform, dim_genre, dim_developer, dim_publisher) -> DataFrame:
    """
    FACT_GAME_REVIEW  —  grão: 1 linha por jogo/plataforma do Metacritic (ID).
    Medidas: METASCORE (crítica, 0-100) e USER_SCORE (usuários, 0-10).
    """
    fact = df.join(dim_game.select("GAME_SK", "GAME_ID"), df["ID"] == dim_game["GAME_ID"], "left").drop("GAME_ID")
    fact = attach_sk(fact, dim_platform,  "PLATFORM",  "PLATFORM")
    fact = attach_sk(fact, dim_genre,     "GENRE",     "GENRE")
    fact = attach_sk(fact, dim_developer, "DEVELOPER", "DEVELOPER")
    fact = attach_sk(fact, dim_publisher, "PUBLISHER", "PUBLISHER")
    fact = fact.withColumn(
        "RELEASE_DATE_SK",
        F.coalesce(F.date_format("RELEASE_DATE", "yyyyMMdd").cast("int"), F.lit(UNKNOWN_SK)))
    return (fact.select("GAME_SK", "PLATFORM_SK", "GENRE_SK", "DEVELOPER_SK", "PUBLISHER_SK",
                        "RELEASE_DATE_SK", "METASCORE", "USER_SCORE")
                .withColumn("FILEDATE_GOLD", F.current_timestamp()))


def save_gold(df: DataFrame, table: str) -> None:
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.{table}"))

## Lendo a silver

In [ ]:
silver_games = (spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.metacritic_games")
                .withColumn("RELEASE_DATE", F.expr("try_cast(RELEASE_DATE AS DATE)")))
print("linhas na silver:", silver_games.count())

## Dimensões

In [ ]:
dim_game      = build_dim_game(silver_games)
dim_platform  = build_dimension(silver_games, "PLATFORM",  "PLATFORM")
dim_genre     = build_dimension(silver_games, "GENRE",     "GENRE")
dim_developer = build_dimension(silver_games, "DEVELOPER", "DEVELOPER")
dim_publisher = build_dimension(silver_games, "PUBLISHER", "PUBLISHER")
dim_date      = build_dim_date(silver_games, "RELEASE_DATE")

for df, table in [(dim_game, "dim_game"), (dim_platform, "dim_platform"), (dim_genre, "dim_genre"),
                  (dim_developer, "dim_developer"), (dim_publisher, "dim_publisher"), (dim_date, "dim_date")]:
    save_gold(df, table)
    print(f"OK  {GOLD_SCHEMA}.{table}")

## Fato

In [ ]:
fact = build_fact_game_review(silver_games, dim_game, dim_platform, dim_genre, dim_developer, dim_publisher)
save_gold(fact, "fact_game_review")

# Verificações de integridade
n_silver, n_fact = silver_games.count(), fact.count()
assert n_silver == n_fact, f"a fato deveria ter {n_silver} linhas e tem {n_fact}"
print(f"OK  fact_game_review: {n_fact} linhas (= silver)")
print("jogos sem GAME_SK:", fact.where(F.col("GAME_SK") == UNKNOWN_SK).count())

## Demais tabelas da silver

`steam_metrics`, `regional_sales`, `how_long_to_beat` e `walkthrough_links` ainda precisam entrar no modelo (novas dimensões e/ou novas fatos, ligadas ao jogo pelo nome ou pelo ID). Confira os schemas e modele conforme o grão de cada uma.

In [ ]:
for table in ["steam_metrics", "regional_sales", "how_long_to_beat", "walkthrough_links"]:
    print(f"--- silver.{table}")
    spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.{table}").printSchema()

## Conferindo o resultado

In [ ]:
%sql
SHOW TABLES IN gold

In [ ]:
%sql
-- Exemplo de consulta no esquema estrela: nota média da crítica por plataforma e ano de lançamento
SELECT p.PLATFORM_NAME, d.YEAR, ROUND(AVG(f.METASCORE), 1) AS AVG_METASCORE, COUNT(*) AS QTD_JOGOS
FROM gold.fact_game_review f
JOIN gold.dim_platform p ON p.PLATFORM_SK = f.PLATFORM_SK
JOIN gold.dim_date d     ON d.DATE_SK     = f.RELEASE_DATE_SK
GROUP BY p.PLATFORM_NAME, d.YEAR
ORDER BY d.YEAR DESC, QTD_JOGOS DESC
LIMIT 20